In [1]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd



In [5]:

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()


import requests
import re
from bs4 import BeautifulSoup

def get_www_papers(page_link, conference_name):
    """WSDM 2024 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""

    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("wsdm2024_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 리스트가 포함된 p 태그 찾기
    for section in soup.find_all("p", class_="gutentor-text"):
        elements = list(section.children)  # 모든 자식 요소 탐색

        title_text = None
        authors_raw = ""

        for elem in elements:
            if elem.name == "strong":  # 논문 제목 찾기
                if title_text and authors_raw:  # 이전 논문 저장 후 새로운 논문으로 이동
                    authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw.strip())  # 괄호 안의 기관 정보 제거
                    papers.append({
                        "title": title_text,
                        "authors": authors_cleaned,
                        "pdf_link": None,
                        "code_url": None,
                        "conference_name": conference_name
                    })
                title_text = elem.text.strip()  # 새로운 논문 제목 저장
                authors_raw = ""  # 새로운 저자 정보 초기화

            elif isinstance(elem, str):  # 저자 정보가 텍스트로 포함됨
                authors_raw += " " + elem.strip()

        # 마지막 논문 저장
        if title_text and authors_raw:
            authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw.strip())  # 괄호 안의 기관 정보 제거
            papers.append({
                "title": title_text,
                "authors": authors_cleaned,
                "pdf_link": None,
                "code_url": None,
                "conference_name": conference_name
            })

    return papers


In [6]:
page_link='https://www.wsdm-conference.org/2024/accepted-papers/' 
DB_PATH = "con_db/wsdm_conference_2024.db"
conference_name = 'WSDM 2024'
papers = get_www_papers(page_link, conference_name)

In [8]:
df_papers = pd.DataFrame(papers)

In [9]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Defense Against Model Extraction Attacks on Re...,Sixiao Zhang *; Hongzhi Yin ; Hongxu Chen ; Ch...,None,None,WSDM 2024
1,GPT4Table: Can Large Language Models Understan...,Yuan Sui *; Mengyu Zhou ; Mingjie Zhou ; Shi H...,None,None,WSDM 2024
2,Hierarchical Multimodal Pre-training for Visua...,Hongshen Xu *; Lu Chen ; Zihan Zhao ; Da Ma ; ...,None,None,WSDM 2024
3,Motif-based Prompt Learning for Universal Cros...,Bowen Hao *; Chaoqun Yang ; Lei Guo ; Junliang...,None,None,WSDM 2024
4,"To Copy, or not to Copy; That is a Critical Is...",Haw-shiuan Chang *; Nikhil Agarwal ; Andrew Mc...,None,None,WSDM 2024


In [11]:
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [12]:
save_to_database(df_papers, conference_name=conference_name, DB_PATH=DB_PATH)

109개의 논문이 WSDM 2024에 저장되었습니다.


# 2023 

In [14]:
url = "https://www.wsdm-conference.org/2023/program/accepted-papers"
DB_PATH = "con_db/wsdm_conference_2023.db"
conference_name = 'WSDM 2023'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [17]:
import requests
import re
from bs4 import BeautifulSoup

def get_www_papers(page_link, conference_name):
    """WSDM 2023 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""

    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("wsdm2023_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 리스트가 포함된 div 태그 찾기
    for paper_div in soup.find_all("div", class_="col-12"):
        title_tag = paper_div.find("h4")
        authors_tag = paper_div.find("p")

        if title_tag and authors_tag:
            title_text = title_tag.text.strip()

            # 저자 정보에서 기관명(괄호 속 내용) 제거
            authors_raw = authors_tag.text.strip()
            authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw).strip()

            papers.append({
                "title": title_text,
                "authors": authors_cleaned,
                "pdf_link": None,  # 현재 PDF 링크 없음
                "code_url": None,
                "conference_name": conference_name
            })

    return papers


In [18]:
papers =get_www_papers(url, conference_name)

In [19]:
df_papers = pd.DataFrame(papers)

In [20]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Friendly Conditional Text Generator,Noriaki Kawamae *,None,None,WSDM 2023
1,Friendly Conditional Text Generator,Noriaki Kawamae *,None,None,WSDM 2023
2,An F-shape Click Model for Information Retriev...,Lingyue Fu ; Jianghao Lin *; Weiwen Liu ; Ruim...,None,None,WSDM 2023
3,Towards Universal Cross-Domain Recommendation,Jiangxia Cao *; Shaoshuai Li ; Bowen Yu ; xiao...,None,None,WSDM 2023
4,Simultaneous Linear Multi-view Attributed Grap...,Chakib Fettal *; Lazhar Labiod ; Mohamed Nadif,None,None,WSDM 2023


In [21]:
save_to_database(df_papers, conference_name=conference_name, DB_PATH=DB_PATH)

124개의 논문이 WSDM 2023에 저장되었습니다.


# 2022

In [22]:
url = "https://www.wsdm-conference.org/2022/program/accepted-papers"
DB_PATH = "con_db/wsdm_conference_2022.db"
conference_name = 'WSDM 2022'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [28]:
import requests
import re
from bs4 import BeautifulSoup

def get_www_papers(page_link, conference_name):
    """WSDM 2022 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""

    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("wsdm2022_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []
    current_title = None

    # 논문 리스트 찾기
    for elem in soup.find_all(["h4", "p"]):
        if elem.name == "h4":  # 논문 제목
            if current_title:  # 이전 논문 저장 후 새로운 논문 시작
                papers.append({
                    "title": current_title,
                    "authors": authors_cleaned,
                    "pdf_link": None,
                    "code_url": None,
                    "conference_name": conference_name
                })
            current_title = elem.text.strip()
            authors_raw = ""

        elif elem.name == "p":  # 저자 정보
            authors_raw = elem.text.strip()
            authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw).strip()  # 괄호 안 소속 제거

    # 마지막 논문 저장
    if current_title:
        papers.append({
            "title": current_title,
            "authors": authors_cleaned,
            "pdf_link": None,
            "code_url": None,
            "conference_name": conference_name
        })

    return papers


In [29]:
papers =get_www_papers(url, conference_name)

In [30]:
df_papers = pd.DataFrame(papers)

In [31]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Long Short-Term Temporal Meta-learning in Onli...,Ruobing Xie *; Yalong Wang ; Rui Wang ; Yuanfu...,None,None,WSDM 2022
1,Estimating Causal Effects of Multi-Aspect Onli...,Lu Cheng *; Ruocheng Guo ; Huan Liu,None,None,WSDM 2022
2,A Cooperative-Competitive Multi-Agent Framewor...,Chao Wen *; Miao Xu ; Zhilin Zhang ; ZHENZHE Z...,None,None,WSDM 2022
3,Improving Knowledge Tracing with Collaborative...,Ting Long *; Jiarui Qin ; Jian Shen ; Weinan Z...,None,None,WSDM 2022
4,It Is Different When Items Are Older: Debiasin...,Jin Huang ; Harrie Oosterhuis *; Maarten de Rijke,None,None,WSDM 2022


In [32]:
save_to_database(df_papers,conference_name=conference_name, DB_PATH=DB_PATH)

160개의 논문이 WSDM 2022에 저장되었습니다.


# 2021

In [41]:
import requests
import re
from bs4 import BeautifulSoup

def get_www_papers(page_link, conference_name):
    """WSDM 2021 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""

    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("wsdm2021_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 리스트 찾기
    for section in soup.find_all("div", class_="accepted"):
        for paper in section.find_all("h3"):  # 제목 태그
            title_text = paper.text.strip()
            next_sibling = paper.find_next_sibling("p")  # 저자 태그 찾기

            if next_sibling:
                authors_raw = next_sibling.text.strip()
                authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw).strip()  # 괄호 안 소속 제거

                papers.append({
                    "title": title_text,
                    "authors": authors_cleaned,
                    "pdf_link": None,  # 현재 PDF 링크 없음
                    "code_url": None,
                    "conference_name": conference_name
                })

    return papers

In [42]:
url='https://www.wsdm-conference.org/2021/accepted-papers.php'
DB_PATH = "con_db/wsdm_conference_2021.db"
conference_name = 'WSDM 2021'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [43]:
paper=get_www_papers(url,conference_name=conference_name)

In [44]:
df_paper = pd.DataFrame(paper) 
df_paper.head()

,title,authors,pdf_link,code_url,conference_name
0,235: Population-Scale Study of Human Needs Dur...,"Jina Suh , Eric Horvitz , Ryen White , Tim Alt...",None,None,WSDM 2021
1,465: Ad Delivery Algorithms: The Hidden Arbite...,"Muhammad Ali , Piotr Sapiezynski , Aleksandra ...",None,None,WSDM 2021
2,479: Towards Ordinal Suicide Ideation Detectio...,"Ramit Sawhney , Harshit Joshi , Saumya Gandhi ...",None,None,WSDM 2021
3,493: Semi-Supervised Text Classification via S...,"Payam Karisani , Negin Karisani",None,None,WSDM 2021
4,134: DeepXML: A Deep Extreme Multi-Label Learn...,"Kunal Dahiya , Deepak Saini , Anshul Mittal , ...",None,None,WSDM 2021


In [45]:
save_to_database(df_paper, conference_name, DB_PATH)

111개의 논문이 WSDM 2021에 저장되었습니다.


# 2020

In [46]:
url = 'https://www.wsdm-conference.org/2020/accepted-papers.php'
DB_PATH = "con_db/wsdm_conference_2020.db"
conference_name = 'WSDM 2020'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [47]:
import requests
import re
from bs4 import BeautifulSoup

def get_www_papers(page_link, conference_name):
    """WSDM 2021 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""

    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("wsdm2020_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 리스트 찾기
    for section in soup.find_all("div", class_="accepted"):
        for paper in section.find_all("h3"):  # 제목 태그
            title_text = paper.text.strip()
            next_sibling = paper.find_next_sibling("p")  # 저자 태그 찾기

            if next_sibling:
                authors_raw = next_sibling.text.strip()
                authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw).strip()  # 괄호 안 소속 제거

                papers.append({
                    "title": title_text,
                    "authors": authors_cleaned,
                    "pdf_link": None,  # 현재 PDF 링크 없음
                    "code_url": None,
                    "conference_name": conference_name
                })

    return papers

In [48]:
papers = get_www_papers(url, conference_name=conference_name)

In [49]:
papers

[{'title': 'A Context-Aware Click Model for Web Search',
  'authors': 'Jia Chen, Jiaxin Mao, Yiqun Liu, Min Zhang, Shaoping Ma .',
  'pdf_link': None,
  'code_url': None,
  'conference_name': 'WSDM 2020'},
 {'title': 'Inf-VAE: A Variational Autoencoder Framework to Integrate Homophily and Influence in Diffusion Prediction',
  'authors': 'Aravind Sankar, Xinyang Zhang, Adit Krishnan, Jiawei Han .',
  'pdf_link': None,
  'code_url': None,
  'conference_name': 'WSDM 2020'},
 {'title': 'A Stochastic Treatment of Learning to Rank Scoring Functions',
  'authors': 'Sebastian Bruch, Shuguang Han, Michael Bendersky, Marc Najork .',
  'pdf_link': None,
  'code_url': None,
  'conference_name': 'WSDM 2020'},
 {'title': 'A structural graph representation learning framework',
  'authors': 'Ryan Rossi ;\n\t\tNesreen Ahmed ;\n\t\t Eunyee Koh, Sungchul Kim, Anup Rao ; \n\t\tYasin Abbasi-Yadkori .',
  'pdf_link': None,
  'code_url': None,
  'conference_name': 'WSDM 2020'},
 {'title': 'Ad Close Mitigatio

In [50]:
df_paper = pd.DataFrame(papers)
df_paper.head()

,title,authors,pdf_link,code_url,conference_name
0,A Context-Aware Click Model for Web Search,"Jia Chen, Jiaxin Mao, Yiqun Liu, Min Zhang, Sh...",None,None,WSDM 2020
1,Inf-VAE: A Variational Autoencoder Framework t...,"Aravind Sankar, Xinyang Zhang, Adit Krishnan, ...",None,None,WSDM 2020
2,A Stochastic Treatment of Learning to Rank Sco...,"Sebastian Bruch, Shuguang Han, Michael Benders...",None,None,WSDM 2020
3,A structural graph representation learning fra...,Ryan Rossi ;\n\t\tNesreen Ahmed ;\n\t\t Eunyee...,None,None,WSDM 2020
4,Ad Close Mitigation for Improved User Experien...,"Natalia Silberstein, Oren Somekh, Yair Koren, ...",None,None,WSDM 2020


In [51]:
save_to_database(df_paper, conference_name, DB_PATH)

91개의 논문이 WSDM 2020에 저장되었습니다.


# 2019

In [52]:
url = 'https://www.wsdm-conference.org/2019/accepted-papers.php'
DB_PATH = "con_db/wsdm_conference_2019.db"
conference_name = 'WSDM 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [57]:
import requests
import re
from bs4 import BeautifulSoup

def get_www_papers(page_link, conference_name):
    """WSDM 2021 학회의 Accepted Papers 페이지에서 논문 정보 크롤링"""

    # 웹페이지 요청
    response = requests.get(page_link)
    
    if response.status_code != 200:
        print("❌ 페이지를 불러오지 못했습니다.")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # HTML 저장 (디버깅용)
    with open("wsdm2019_accepted_papers.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    papers = []

    # 논문 리스트 찾기
    for section in soup.find_all("div", class_="accepted"):
        for paper in section.find_all("h3"):  # 제목 태그
            title_text = paper.text.strip()
            next_sibling = paper.find_next_sibling("p")  # 저자 태그 찾기

            if next_sibling:
                authors_raw = next_sibling.text.strip()

                # 괄호 안 소속 정보 제거 + ; 와 개행 문자 제거
                authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw)
                authors_cleaned = re.sub(r";\s*\n", ", ", authors_cleaned)  # ;\n 제거 후 , 로 변환
                authors_cleaned = re.sub(r"\s+", " ", authors_cleaned).strip()  # 불필요한 공백 정리

                papers.append({
                    "title": title_text,
                    "authors": authors_cleaned,
                    "pdf_link": None,  # 현재 PDF 링크 없음
                    "code_url": None,
                    "conference_name": conference_name
                })

    return papers


In [58]:
papers = get_www_papers(url, conference_name)

In [59]:
papers

[{'title': 'A General View for Network Embedding as Matrix Factorization.',
  'authors': 'Xin Liu , Tsuyoshi Murata , Kyoung-Sook Kim , Chatchawan Kotarasu , Chenyi Zhuang .',
  'pdf_link': None,
  'code_url': None,
  'conference_name': 'WSDM 2019'},
 {'title': 'All Those Wasted Hours: On Task Abandonment in Crowdsourcing.',
  'authors': 'Lei Han , Kevin Roitero , Ujwal Gadiraju , Cristina Sarasua , Alessandro Checco , Eddy Maddalena , Gianluca Demartini .',
  'pdf_link': None,
  'code_url': None,
  'conference_name': 'WSDM 2019'},
 {'title': 'A Sequential Test for Selecting the Better Variant: Online A/B testing, Adaptive Allocation and Continuous Monitoring.',
  'authors': 'Nianqiao Ju , Diane Hu, Adam Henderson, Liangjie Hong .',
  'pdf_link': None,
  'code_url': None,
  'conference_name': 'WSDM 2019'},
 {'title': 'A Simple But Effective Generative Model for Next Item Recommendation.',
  'authors': 'Fajie Yuan , Alexandros Karatzoglou, Ioannis Arapakis , Joemon Jose , Xiangnan He .'

In [60]:
df_papers = pd.DataFrame(papers) 
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,A General View for Network Embedding as Matrix...,"Xin Liu , Tsuyoshi Murata , Kyoung-Sook Kim , ...",None,None,WSDM 2019
1,All Those Wasted Hours: On Task Abandonment in...,"Lei Han , Kevin Roitero , Ujwal Gadiraju , Cri...",None,None,WSDM 2019
2,A Sequential Test for Selecting the Better Var...,"Nianqiao Ju , Diane Hu, Adam Henderson, Liangj...",None,None,WSDM 2019
3,A Simple But Effective Generative Model for Ne...,"Fajie Yuan , Alexandros Karatzoglou, Ioannis A...",None,None,WSDM 2019
4,A State Transition Model for Mobile Notificati...,"Yiping Yuan , Jing Zhang , Shaunak Chatterjee,...",None,None,WSDM 2019


In [61]:
save_to_database(df_papers, conference_name, DB_PATH)

84개의 논문이 WSDM 2019에 저장되었습니다.


# 2018

In [65]:
url = 'https://www.wsdm-conference.org/2018/accepted-papers.html'
DB_PATH = "con_db/wsdm_conference_2018.db"
conference_name = 'WSDM 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [71]:
import re
from bs4 import BeautifulSoup
import pandas as pd

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 WSDM 2018 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 제목과 저자 정보가 포함된 리스트 찾기
    for li in soup.find_all("li"):
        strong_tag = li.find("strong")  # 논문 제목이 포함된 태그 찾기
        if strong_tag:
            title_text = strong_tag.text.strip()  # 논문 제목 추출

            # 제목 이후의 텍스트가 저자 정보라고 가정
            author_text = li.text.replace(title_text, "").strip()
            author_text = re.sub(r"\(.*?\)", "", author_text)  # 기관명 제거
            author_text = re.sub(r";\s*\n", ", ", author_text)  # `;\n` 제거 후 쉼표로 정리
            author_text = re.sub(r"\s+", " ", author_text).strip()  # 불필요한 공백 정리

            papers.append({
                "title": title_text,
                "authors": author_text,
                "pdf_link": None,  # PDF 링크 없음
                "code_url": None,
                "conference_name": conference_name
            })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [72]:
paper = get_www_papers('/home/cvlab/papers-update/conference/wsdm2018_accepted_papers.html', conference_name)

In [73]:
paper.head()

,title,authors,pdf_link,code_url,conference_name
0,Cognitive Biases in Crowdsourcing,Carsten Eickhoff,None,None,WSDM 2018
1,DSANLS: Accelerating Distributed Nonnegative M...,Yuqiu Qian ; Conghui Tan ; Nikos Mamoulis ; Da...,None,None,WSDM 2018
2,Index Compression Using Byte-Aligned ANS Codin...,Alistair Moffat ; Matthias Petri,None,None,WSDM 2018
3,Ballpark Crowdsourcing: The Wisdom of Rough Gr...,Tom Hope ; Dafna Shahaf,None,None,WSDM 2018
4,Micro Behaviors: A New Perspective in E-commer...,Meizi Zhou ; Zhuoye Ding ; Jiliang Tang ; Dawe...,None,None,WSDM 2018


In [75]:
save_to_database(paper, conference_name=conference_name, DB_PATH=DB_PATH)

81개의 논문이 WSDM 2018에 저장되었습니다.
